# 6교시 · 관계를 보는 법, 그리고 분석 에이전트

**이 시간은 두 부분입니다.**

### Part 1 — 두 숫자의 관계
상관계수로 관계를 재고, **상관과 인과가 왜 다른지** 확인합니다.

### Part 2 — 분석 에이전트 만들기
질문을 던지면 코드를 써서 답하는 도구를 **직접 만듭니다.**

> 빈칸 문제는 없습니다. 셀을 위에서부터 실행하면서 눈으로 확인하면 됩니다.

---
# 준비

먼저 필요한 것들을 불러옵니다.

In [ ]:
!pip -q install langchain langchain-openai

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders = pd.read_csv(BASE + 'superstore_orders.csv',
                     parse_dates=['Order Date', 'Ship Date'])

print(orders.shape)

## 그래프에 한글이 나오게 하기

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

## API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. **강사가 알려 주는 키**를 붙여 넣으세요.

`getpass` 를 쓰면 입력한 값이 **화면에 보이지 않고 노트북에도 저장되지 않습니다.**

In [ ]:
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('API 키를 붙여 넣으세요: ')

print('연결 준비 완료')

## 오늘 쓰는 데이터 — Superstore 주문 내역

미국의 문구·가구 유통사 주문 내역입니다.

- **한 행 = 주문에 담긴 품목 하나** (주문 하나에 품목이 여럿이면 여러 행)
- 기간: 2023년 ~ 2026년, 약 1만 행
- 주요 열 — `Order ID`(주문번호) · `Order Date`(주문일) · `Region`(지역) ·
  `Category`(대분류) · `Sales`(매출) · `Quantity`(수량) ·
  `Discount`(할인율) · `Profit`(이익) · `Customer ID`(고객번호)

---
# Part 1 · 두 숫자의 관계

# 1-1. 상관계수

지금까지는 **집단끼리 비교**했습니다.
이번에는 **두 숫자가 같이 움직이는가**를 봅니다.

`corr()` 한 줄이면 나옵니다.

In [ ]:
print(round(orders['Discount'].corr(orders['Profit']), 3))

**-0.219** 가 나왔습니다. 이 숫자를 읽는 법입니다.

상관계수는 **-1 에서 +1 사이**의 값입니다.

| 값 | 뜻 |
|---|---|
| **+1 에 가깝다** | 한쪽이 오르면 다른 쪽도 오른다 |
| **0 에 가깝다** | 직선 관계가 약하다 &mdash; *관계가 없다는 뜻은 아닙니다* |
| **-1 에 가깝다** | 한쪽이 오르면 다른 쪽은 내린다 |

-0.219 는 **음수이지만 -1 에서 한참 멀어** 관계가 약합니다.
할인율만으로 이익이 정해지지는 않는다는 뜻입니다.

In [ ]:
print(orders[['Sales', 'Quantity', 'Discount', 'Profit']].corr().round(3))

`corr()` 를 표 전체에 걸면 **한 번에** 볼 수 있습니다.
대각선이 1인 것은 자기 자신과의 상관이기 때문입니다.

# 1-2. 숫자만 보지 말고 그림도 봅니다

같은 상관계수라도 **점이 흩어진 모양은 아주 다를 수 있습니다.**

In [ ]:
subset = orders[orders['Profit'].between(-500, 500)]

subset.plot(kind='scatter', x='Discount', y='Profit', alpha=0.15, figsize=(8, 4))
plt.axhline(0, color='red', linewidth=1)
plt.title('할인율과 이익')
plt.show()

할인율이 올라갈수록 **아래로 처지는 경향**이 보입니다.
다만 같은 할인율에서도 이익이 넓게 퍼져 있습니다. 이것이 &ldquo;약한 상관&rdquo;의 실제 모습입니다.

> **&ldquo;할인율 30%인 주문의 이익은 얼마입니까?&rdquo;**
> 이 질문에는 답할 수 없습니다. 전체 경향은 있지만 개별 주문은 맞히지 못합니다.

# 1-3. 상관은 인과가 아닙니다

할인율과 이익은 음의 관계입니다.
그렇다고 **&ldquo;할인을 없애면 이익이 오른다&rdquo;** 고 말할 수 있을까요?

**말할 수 없습니다.** 이유가 둘 있습니다.

| | 무엇 | 이 경우라면 |
|---|---|---|
| **순서가 반대** | 이익이 할인을 부른 것 | 원래 안 팔리는 품목이라 할인을 했다 |
| **숨은 원인** | 제3의 것이 둘을 함께 움직임 | 재고가 오래된 품목이라 할인도 하고 이익도 낮다 |

두 경우 모두 **할인을 없애도 이익은 오르지 않습니다.**

> <mark>같이 움직인다고 해서 한쪽이 다른 쪽의 원인인 것은 아닙니다.</mark>

## 그럼 인과는 어떻게 아나

원칙적으로는 **다른 조건을 같게 만들고 한 가지만 바꿔** 봐야 합니다.
주문을 무작위로 두 무리로 나눠 한쪽만 할인하는 방식입니다 &mdash; **A/B 테스트**라고 부릅니다.

가진 데이터를 들여다보는 것만으로는 여기까지 갈 수 없습니다.

---
# Part 2 · 분석 에이전트 만들기

---
# 2-1. 에이전트란 무엇인가

**에이전트(agent)** 는 이렇게 이루어집니다.

> **에이전트 = LLM + 하니스(harness)**

**LLM** 은 말을 알아듣고 글을 쓰는 부분입니다. 그것만으로는 아무것도 하지 못합니다.
읽지도, 계산하지도, 저장하지도 못합니다.

**하니스**는 LLM 이 실제로 일을 하게 붙여 주는 장치입니다. 대개 이런 것들이 들어갑니다.

| 하니스의 구성 | 하는 일 |
|---|---|
| **도구 (tool)** | 파일 읽기, 계산, 검색, 코드 실행처럼 **실제 행동** |
| **기억 (memory)** | 앞에서 무슨 이야기를 했는지 |
| **맥락 (context)** | 지금 다루는 자료가 무엇인지 |
| **반복 (loop)** | 결과를 보고 다시 시도하기 |

오늘은 이 중에서 **도구 하나만** 붙입니다 — **&ldquo;파이썬 코드를 실행한다&rdquo;** 입니다.
거기에 **맥락**으로 표의 구조를 넣어 주면, 그것만으로도 쓸 만한 분석 에이전트가 됩니다.

---
# 2-2. 도구를 만듭니다

에이전트에게 붙일 **도구**를 먼저 만듭니다.
하는 일은 하나입니다 &mdash; **파이썬 코드를 받아서 실행하고, 출력을 돌려준다.**

## 그 전에 &mdash; 함수(function)

오늘 처음으로 `def` 가 나옵니다. 잠깐만 보고 가겠습니다.

**함수는 코드에 이름을 붙여 두는 것**입니다.
지금까지는 필요할 때마다 코드를 그 자리에 썼습니다. 함수를 만들면
그 코드 뭉치에 이름이 생기고, **이름만 부르면 언제든 다시 실행**됩니다.

```python
def 이름(받을 값):
    ...하는 일...
    return 돌려줄 값
```

| 부분 | 뜻 |
|---|---|
| `def` | &ldquo;이제부터 함수를 하나 만든다&rdquo; |
| `이름(...)` | 함수 이름과, **받을 값**(매개변수) |
| 들여쓴 줄 | 함수가 하는 일 &mdash; **들여써야** 함수 안이 됩니다 |
| `return` | **돌려줄 값**. 부른 자리로 이 값이 돌아갑니다 |

사실 우리는 계속 함수를 **써** 왔습니다. `print()`, `len()`, `round()` 가 전부 함수입니다.
이번에는 쓰는 대신 **직접 하나 만드는** 것뿐입니다.

> 도구는 반드시 함수여야 합니다. 에이전트가 &ldquo;이 도구를 써라&rdquo;고 결정하면
> **그 함수를 대신 불러 주는** 방식으로 동작하기 때문입니다.


In [ ]:
import io, contextlib


def run_python(code: str) -> str:
    """pandas 코드를 실행하고 출력을 돌려준다. 표 이름은 orders."""
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            exec(code, {'pd': pd, 'plt': plt, 'orders': orders})
    except Exception as e:
        return f'오류: {type(e).__name__}: {e}'
    return buffer.getvalue() or '(출력 없음)'


print(run_python("print(orders.shape)"))

방금 만든 함수를 도구의 관점에서 다시 봅니다.

| 함수의 이 부분 | 도구에서는 |
|---|---|
| 함수 이름 `run_python` | AI 가 부를 **도구 이름** |
| `"""..."""` 설명글 | AI 가 읽는 **도구 설명** &mdash; 언제 쓸지 여기서 판단합니다 |
| `code: str` | 도구에 **무엇을 넣어야 하는지** |
| `return` 한 값 | AI 에게 **돌아가는 결과** |

**그래서 설명글이 중요합니다.** LangChain 이 그 설명을 읽어서
**AI 에게 &ldquo;이런 도구가 있다&rdquo;고 알려 줍니다.** 설명이 부실하면 AI 가 도구를 안 씁니다.

오류가 나도 그대로 문자열로 돌려줍니다.
그래야 AI 가 **무엇이 잘못됐는지 보고 다시 시도**할 수 있습니다.


---
# 2-3. 데이터를 통째로 보내지는 않습니다

회사 데이터는 대개 **밖으로 내보낼 수 없습니다.**

그래서 표 전체를 올리는 대신, **데이터가 어떻게 생겼는지**만 보냅니다.
`dtypes` 로 열 이름과 자료형만 뽑으면 됩니다.


In [ ]:
schema = "\n".join("- {} ({})".format(k, v)
                     for k, v in orders.dtypes.astype(str).items())

print(schema)


**여기에는 실제 값이 하나도 없습니다.** 열 이름과 자료형뿐입니다.

이것만 보내도 AI 는 어떤 코드를 써야 할지 압니다.
1교시에서 배운 **&ldquo;데이터를 열면 무엇부터 보는가&rdquo;** 가 그대로 쓰이는 셈입니다.

> **다만 &ldquo;아무것도 안 나간다&rdquo;는 뜻은 아닙니다.**
> 표를 통째로 올리지 않을 뿐이고, **계산 결과는 나갑니다.**
> 왜 그런지는 조금 뒤 2-5 절에서 눈으로 확인합니다.


---
# 2-4. 에이전트 만들기

이제 **LangChain** 의 `create_agent` 로 묶습니다. 넣을 것은 셋입니다.

| 넣는 것 | 무엇인가 |
|---|---|
| `model` | 어떤 LLM 을 쓸지 |
| `tools` | 붙일 **도구** 목록 |
| `system_prompt` | 미리 알려 줄 **맥락** |

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.6-luna", reasoning_effort="none")

instructions = f"""너는 pandas 분석 도우미다.
표 이름은 orders, 행 수는 {len(orders):,}. 열은 다음과 같다:
{schema}

질문을 받으면 run_python 도구로 코드를 돌려 답하라.
pd 와 orders 는 이미 있다. 결과는 반드시 print() 하라."""

agent = create_agent(model=llm, tools=[run_python], system_prompt=instructions)

print("에이전트 준비 완료")

**이것이 전부입니다.** 우리가 만든 것은 도구 하나(`run_python`)와 맥락(`schema`)뿐이고,
나머지는 `create_agent` 가 알아서 합니다 &mdash;
질문을 LLM 에 보내고, 도구를 부르고, 결과를 다시 LLM 에 넘겨 답을 만드는 **반복**까지요.

---
# 2-5. 물어봅니다

이제 말로 물어보면 됩니다. 그런데 `agent.invoke(...)` 가 돌려주는 것은
**메시지 목록**이라, 그대로 출력하면 읽기가 어렵습니다.
오간 내용을 &ldquo;AI 가 쓴 코드 / 실행 결과 / 답변&rdquo; 으로 나눠 찍어야 볼 만합니다.

그 정리 코드를 **질문할 때마다 다시 쓸 수는 없습니다.** 아래에서 열 번 넘게 물어볼 거니까요.
이럴 때가 함수로 묶을 자리입니다 &mdash; **같은 코드를 두 번 넘게 쓰게 될 때.**

`ask` 라는 이름을 붙여 두면, 앞으로는 `ask("...")` 한 줄로 끝납니다.


In [ ]:
def ask(question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    for m in result["messages"][1:]:
        if m.type == "ai" and m.tool_calls:
            print("[AI 가 쓴 코드]")
            print(m.tool_calls[0]["args"]["code"])
            print()
        elif m.type == "tool":
            print("[실행 결과]")
            print(m.content)
            print()
        elif m.type == "ai" and m.content:
            print("[답변]")
            print(m.content)


In [ ]:
ask("지역별 매출 합계를 큰 순서로 알려 줘")

In [ ]:
ask("주문은 실제로 몇 건이야?")

두 번째 답을 잘 보세요. `nunique()` 를 썼습니다.

**한 행이 주문 하나가 아니라는 것**을 AI 도 구조만 보고 알아챈 것입니다.
1교시에서 우리가 직접 확인했던 그 내용입니다.

## 잠깐 &mdash; 방금 무엇이 밖으로 나갔나

위 출력의 **`[실행 결과]` 블록이 곧 AI 에게 전송된 내용**입니다.

`run_python` 이 돌려준 값은 화면에만 찍히는 게 아닙니다.
AI 가 그 숫자를 보고 답을 써야 하므로 **다시 프롬프트에 붙어 모델로 올라갑니다.**

```
질문 → AI 가 코드 작성 → run_python 실행 → 출력을 AI 에게 되돌려 줌 → 답변
                                            ↑ 여기서 나갑니다
```

| | 나가나 |
|---|---|
| 표 전체 (1만 행) | **아니오** |
| 열 이름 · 자료형 | 예 (`schema`) |
| **집계 결과 · 검정 결과** | **예** |
| **AI 가 `print()` 한 것은 무엇이든** | **예** |

마지막 줄이 중요합니다. 코드를 쓰는 쪽이 AI 라서 **무엇이 나갈지 우리가 정하지 못합니다.**
AI 가 `print(orders.head(50))` 이나 `print(orders['Customer ID'].unique())` 를
써야겠다고 판단하면 **원본 행과 고객 ID 가 그대로 나갑니다.** 지금 `run_python` 에는 이를 막는 장치가 없습니다.

> **사내에 쓰기 전에 확인할 것**
> ① 결과에 개인정보나 식별자가 섞이는가 &nbsp; ② 어느 회사의 어느 모델로, 어느 리전에 전송되는가
> &nbsp; ③ 그 데이터가 학습에 쓰이지 않는다는 계약이 있는가
>
> &ldquo;표를 안 올렸으니 괜찮다&rdquo; 는 판단은 **틀립니다.**


In [ ]:
ask("평균 주문 금액은 얼마야?")

여기서도 `Order ID` 로 먼저 묶은 뒤 평균을 냈습니다.
그냥 `Sales.mean()` 을 했다면 **주문 평균이 아니라 품목 평균**이 나왔을 것입니다.

---
# 2-6. AI 가 거부할 때

없는 것을 물어보면 어떻게 될까요?

In [ ]:
ask("반품률이 가장 높은 지역은?")

**이 표에는 반품 정보가 없습니다.** 그래서 AI 가 답을 만들지 못한다고 알려 줍니다.

> **거부는 나쁜 신호가 아닙니다.**
> **&ldquo;내가 잘못 물었거나, 이 데이터로는 답할 수 없다&rdquo;** 는 뜻입니다.

3교시에서 반품 정보는 **다른 표**에 있었다는 것을 떠올려 보세요.
붙여야 답할 수 있는 질문입니다.

In [ ]:
ask("이 데이터로 광고 효과를 알 수 있어?")

광고비도 이 표에 없습니다.

1교시의 **&ldquo;이 데이터로 답할 수 없는 질문&rdquo;** 이 여기서 다시 나옵니다.
에이전트를 쓰면 **무엇이 없는지**를 더 빨리 알게 됩니다.

---
# 2-7. 코드는 맞는데, 해석이 틀릴 때

여기가 오늘 가장 중요한 부분입니다.

In [ ]:
ask("고객 재구매율은?")

**98.5%** 가 나왔습니다. 코드도 맞고 계산도 맞습니다.

그런데 이대로 **&ldquo;우리 재구매율은 98%입니다&rdquo;** 라고 보고해도 될까요?

직접 확인해 봅니다.

In [ ]:
order_counts = orders.groupby('Customer ID')['Order ID'].nunique()

print('고객 수          :', len(order_counts))
print('2건 이상 산 고객 :', (order_counts > 1).sum())
print('고객당 평균 주문 : {:.1f}건'.format(order_counts.mean()))
print()
print('기간: ', orders['Order Date'].min().date(), '~', orders['Order Date'].max().date())

## 무엇이 문제였나

**4년치 데이터**입니다. 4년 동안 두 번 이상 산 사람이 98.5%인 것은 당연합니다.
고객당 평균 주문이 이미 6건이 넘습니다.

> 이 숫자는 **&ldquo;재구매율&rdquo;이 아니라 &ldquo;4년 동안 2번 이상 산 사람의 비율&rdquo;** 입니다.
> 재구매율을 말하려면 **기간을 정해야** 합니다 &mdash; 한 달 안에, 석 달 안에처럼요.

**AI 는 이것을 짚어 주지 못했습니다.**
데이터가 몇 년치인지, 이 회사에서 &ldquo;재구매&rdquo;를 어떻게 정의하는지 모르기 때문입니다.

<mark>코드가 맞다고 답이 맞는 것은 아닙니다.</mark>

---
# 2-8. 그래서 무엇을 알고 있어야 하나

오늘 본 세 가지 상황을 정리하면 이렇습니다.

| 상황 | AI | 사람이 해야 할 일 |
|---|---|---|
| 잘 되는 질문 | 코드를 정확히 씀 | **무엇을 물을지 정한다** |
| 거부하는 질문 | 없다고 알려 줌 | **어디에 있는지 안다** (다른 표에 있는지) |
| **틀린 해석** | 알아채지 못함 | **숫자가 말이 되는지 본다** |

세 번째를 알아채려면 **오늘 배운 것이 다 필요합니다.**

- 1교시 &mdash; 한 행이 무엇인지, 기간이 언제부터인지
- 2교시 &mdash; 무엇이 빠져 있는지
- 3교시 &mdash; 어느 표에 무엇이 있는지
- 4교시 &mdash; 평균 하나를 믿어도 되는지
- 5교시 &mdash; 이 그림이 오해를 부르지 않는지

> **에이전트는 코드를 대신 써 줍니다. 판단을 대신해 주지는 않습니다.**

---
# 2-9. 좋은 질문 쓰는 법

에이전트가 잘 답하게 하려면 **질문을 잘 써야** 합니다.
회사 데이터를 못 올리는 상황에서 챙길 것은 다섯 가지입니다.

| | 무엇을 | 예 |
|---|---|---|
| **①** | **상황** | &ldquo;유통사 영업기획입니다. 할인 정책을 바꿀지 검토 중입니다&rdquo; |
| **②** | **알고 싶은 것** | &ldquo;할인한 주문이 그렇지 않은 주문보다 이익이 낮은지&rdquo; |
| **③** | **데이터 형태** | &ldquo;한 행이 주문 품목 하나입니다. 1만 행. 이익(숫자), 할인율(숫자)&rdquo; |
| **④** | **비교 대상** | &ldquo;할인함 / 안 함 두 집단. 서로 다른 주문입니다&rdquo; |
| **⑤** | **제약** | &ldquo;보안 때문에 데이터는 올릴 수 없습니다. 파이썬 pandas 로 작업합니다&rdquo; |

우리가 만든 에이전트는 **③을 자동으로** 넣어 주고 있었습니다 (`system_prompt` 안의 `schema`).
나머지는 사람이 씁니다.

그리고 **&ldquo;이 방법이 맞지 않게 되는 경우&rdquo;** 를 꼭 함께 물어보세요.
AI 는 대체로 자신 있게 답하므로, 이 질문을 해야 스스로 조건을 붙입니다.

In [ ]:
ask("할인한 주문과 하지 않은 주문의 건당 이익을 비교하고, "
        "이 비교에서 주의할 점도 함께 알려 줘")

---
# 2-10. 프롬프트 모음 &mdash; 붙여넣고 결과를 보세요

아래는 **미리 써 둔 질문들**입니다. 셀을 그대로 실행하면 됩니다.

- **A. 오늘 배운 것을 시켜 봅니다** (①~④) &mdash; 1~4교시에서 손으로 했던 일을 말로 시킵니다
- **B. 안 배운 것도 시켜 봅니다** (⑤~⑨) &mdash; **방법을 모르는 채로** 물어봅니다

각 질문에는 **&ldquo;해석까지 해 달라&rdquo;** 는 요구가 들어 있습니다.
숫자만 받으면 쓸모가 없기 때문입니다. 그리고 **AI 의 해석이 맞는지 판단하는 것은 여러분 몫**입니다.

> 실행할 때마다 AI 가 쓰는 코드는 조금씩 달라집니다. 답이 매번 똑같지 않아도 정상입니다.


---
# A. 오늘 배운 것을 시켜 봅니다

1~4교시에서 **직접 코드를 써서** 했던 일입니다. 이번에는 말로만 시켜 봅니다.
**손으로 해 봤기 때문에** AI 가 제대로 했는지 알아볼 수 있습니다.


## ① 1교시 &mdash; 이 데이터가 무엇인가

데이터를 열면 **가장 먼저 확인하던 것**들입니다.


In [ ]:
ask("이 데이터가 어떤 데이터인지 파악하고 싶어. "
    "행과 열이 몇 개인지, 기간이 언제부터 언제까지인지, "
    "그리고 한 행이 무엇 하나를 뜻하는지 확인해서 알려 줘")


**&ldquo;한 행 = 주문 하나가 아니다&rdquo;** 를 짚었는지 보세요.
`Order ID` 가 중복된다는 걸 확인해야 알 수 있는 내용입니다.


## ② 2교시 &mdash; 이 데이터를 믿어도 되는가

전처리하기 전에 **어디가 망가져 있는지** 보던 단계입니다.


In [ ]:
ask("분석을 시작하기 전에 이 데이터가 깨끗한지 점검하고 싶어. "
    "빠진 값, 중복된 행, 말이 안 되는 값이 있는지 확인하고 "
    "각각 몇 건인지, 그리고 어떻게 처리하는 게 좋을지 알려 줘")


2교시에서 우리가 찾았던 것과 같은지 비교해 보세요.
**처리 방법까지 물었다**는 점이 중요합니다. 찾는 것보다 **어떻게 할지가 판단**입니다.


## ③ 3교시 &mdash; 묶어서 비교하기

`groupby` 로 하던 일입니다.
숫자를 내는 것보다 **&ldquo;그래서 큰 차이인가&rdquo;** 를 묻는 게 중요합니다.


In [ ]:
ask("지역별 이익률(이익 합 ÷ 매출 합)을 구해 큰 순서로 보여 줘. "
    "그리고 1등과 꼴찌의 차이가 실무적으로 의미 있는 수준인지도 함께 말해 줘")


## ④ 4교시 &mdash; 평균을 믿어도 되는가

4교시의 핵심이었습니다. **평균 하나만 보면 속습니다.**


In [ ]:
ask("주문 금액(Sales)의 평균만 보고 '보통 이 정도 산다'고 말해도 되는지 "
    "확인하고 싶어. 필요한 값들을 뽑아서 보여 주고, "
    "평균을 그대로 쓰면 안 되는 이유가 있다면 무엇인지 말해 줘")


**중앙값**과 **분포의 치우침**을 짚었는지 보세요.

평균은 **239원**인데 중앙값은 **54원** 입니다. **네 배 넘게** 차이 납니다.
절반이 넘는 주문이 54원 아래인데, 소수의 큰 주문이 평균을 끌어올린 것입니다.

> &ldquo;보통 이 정도 산다&rdquo; 에 가까운 값은 **평균이 아니라 중앙값**입니다.
> 4교시의 **&ldquo;이 평균을 믿어도 되는가&rdquo;** 가 그대로 나왔습니다.


---
# B. 안 배운 것도 시켜 봅니다

여기서부터는 **우리가 방법을 모르는 것**을 물어봅니다.
&ldquo;어떻게 하는지&rdquo; 대신 **&ldquo;무엇을 알고 싶은지&rdquo;** 만 말하면 됩니다.


## ⑤ 우연인가, 진짜 차이인가

산점도에서 **할인할수록 이익이 처지는** 경향을 봤습니다.
그런데 그 정도 차이는 **아무 이유 없이도 생길 수 있습니다.** 어떻게 가려낼까요?

이런 걸 가려내는 방법을 **통계 검정**이라고 합니다.
종류가 여럿이고 상황마다 쓰는 게 다른데, **어떤 걸 써야 하는지 우리는 모릅니다.**

그래도 괜찮습니다. 대신 **왜 그 방법인지 반드시 되묻습니다.**


In [ ]:
ask("할인한 주문이 그렇지 않은 주문보다 이익이 정말 낮은 건지, "
    "아니면 우연히 그렇게 보이는 건지 확인하고 싶어. "
    "적절한 방법을 네가 골라서 확인해 주고, "
    "① 왜 그 방법을 골랐는지 ② 그 방법이 전제하는 조건 "
    "③ 평균 차이가 실제로 얼마인지 를 함께 알려 줘")


### AI 가 무엇을 골랐는지 보세요

아마 **t-검정**을 골랐을 겁니다. 두 집단의 평균을 비교하는 방법입니다.
이름은 몰라도 됩니다. **&ldquo;두 무리의 평균을 비교하는 상황&rdquo;** 이라 그걸 골랐다는 것만 보면 됩니다.

같이 나온 **p값** 을 읽는 법입니다.

| | |
|---|---|
| **p값이 작다** | &ldquo;차이가 없다면 이런 결과가 나오기 어렵다&rdquo; |
| **p값이 작다 ≠ 차이가 크다** | 데이터가 많으면 **아주 작은 차이도** p값이 작아집니다 |
| **p값이 작다 ≠ 원인이다** | 1-3 절에서 본 그대로입니다 |

그래서 ③ **&ldquo;평균 차이가 실제로 얼마인가&rdquo;** 를 같이 물은 것입니다.
p값만 보고는 **그게 큰 차이인지 알 수 없습니다.**


## ⑥ 이번엔 숫자가 아니라 분류입니다

&ldquo;지역마다 **팔리는 물건 종류**가 다른가?&rdquo;

⑤와 달리 평균을 낼 수 있는 숫자가 아닙니다. 지역도 분류, 카테고리도 분류입니다.
**상황이 다르면 방법도 달라집니다.** 역시 AI 에게 고르게 합니다.


In [ ]:
ask("지역마다 팔리는 카테고리 구성이 다른지 알고 싶어. "
    "적절한 방법을 골라 확인해 주고, 왜 그 방법인지도 말해 줘. "
    "그리고 결과를 통계를 모르는 팀장에게 보고하듯 두세 문장으로 설명해 줘")


**p값이 크게 나왔을 겁니다** (0.4 언저리).

&ldquo;유의하지 않다&rdquo; 는 **실패가 아니라 답**입니다.
> **지역이 달라도 팔리는 카테고리 구성은 비슷하다.**

⑤에서는 p값이 아주 작았는데 여기서는 크게 나왔습니다.
**p값이 늘 작게 나오는 게 아니라는 것**을 여기서 확인하고 가세요.

그리고 ⑤와 ⑥에서 **AI 가 서로 다른 방법을 골랐다**는 점을 보세요.
우리가 한 일은 **무엇을 알고 싶은지 말한 것**뿐입니다.


## ⑦ 함정이 있는 질문

Part 1 에서 상관계수 **-0.219** 를 봤습니다.
&ldquo;약한 관계&rdquo; 라고 읽었는데, 이게 **우연일 가능성**은 없을까요?


In [ ]:
ask("할인율이 높을수록 이익이 낮아지는 관계가 있는지, "
    "그 관계가 우연으로 보기 어려운 수준인지 확인해 줘. "
    "그리고 그 결과가 '할인이 이익을 떨어뜨린다'는 뜻인지 아닌지도 "
    "왜 그런지와 함께 설명해 줘")


**AI 의 답을 채점해 보세요.**

- 상관과 인과를 구분해서 말했습니까?
- **&ldquo;순서가 반대일 수 있다&rdquo;**(안 팔려서 할인했다)를 짚었습니까?
- **&ldquo;숨은 원인&rdquo;**(오래된 재고)의 가능성을 말했습니까?

셋을 다 짚었다면 좋은 답입니다. 하나라도 빠졌다면 **여러분이 채워야 할 자리**입니다.

> p값은 **&ldquo;우연이 아니다&rdquo;** 까지만 말해 줍니다.
> **&ldquo;그래서 원인이다&rdquo;** 는 통계가 아니라 **사람이 하는 주장**입니다.


## ⑧ 사람이 알려 줘야 하는 것

⑤~⑦ 에서는 AI 가 방법을 잘 골랐습니다. 그런데 **모든 검정에는 공통 전제**가 하나 있습니다.
&ldquo;각 행이 서로 독립&rdquo; 이라는 것입니다.

이 표는 **한 행이 주문이 아니라 품목**입니다. 한 주문에서 나온 여러 행은 독립이 아닙니다.
**이건 데이터를 아는 사람만 알려 줄 수 있습니다** &mdash; ①에서 확인했던 그 내용입니다.


In [ ]:
ask("동부와 서부 중 어디가 건당 더 비싸게 파는지, "
    "그 차이가 우연 수준인지 확인해 줘. "
    "참고로 이 표는 한 행이 '주문'이 아니라 '주문에 담긴 품목 하나'야. "
    "그 점이 이 결과를 어떻게 흔들 수 있는지도 함께 말해 줘")


**이것도 유의하지 않게 나옵니다** (p 가 0.3 언저리).

건당 평균이 한쪽이 높게 나오더라도,
**&ldquo;동부가 서부보다 비싸게 판다&rdquo; 고 말할 근거는 없다**는 뜻입니다.
차이가 이 정도면 우연으로도 충분히 나옵니다.

그리고 AI 가 **독립성 전제**를 짚었는지 보세요.
짚었다면 우리가 **&ldquo;한 행이 품목이다&rdquo;** 라고 알려 줬기 때문입니다.
알려 주지 않으면 대개 그냥 검정만 하고 끝냅니다.

**2-9 절의 ③ &ldquo;데이터 형태&rdquo; 를 왜 써야 하는지**가 여기서 드러납니다.


## ⑨ 5교시 &mdash; 그림까지, 결론이 제목에 오게

**&ldquo;이름표 제목이 아니라 결론 제목&rdquo;** 을 그대로 시킵니다.


In [ ]:
ask("월별 매출 추이를 선그래프로 그려 줘. 한글 폰트는 이미 설정돼 있다. "
    "제목은 '월별 매출' 같은 이름표가 아니라, "
    "그림에서 읽히는 결론을 한 문장으로 적어 줘")


---
### 여기까지 하고 나면

검정 이름을 외울 필요는 없습니다. **상황을 말하면 방법은 따라옵니다.**

| 알고 싶은 것 | AI 가 고른 것 |
|---|---|
| 두 무리의 **평균**이 다른가 (⑤⑧) | t-검정 |
| 두 **분류**가 서로 관계있나 (⑥) | 카이제곱 검정 |
| 두 **숫자**가 같이 움직이나 (⑦) | 상관계수와 그 p값 |

| AI 가 잘한 것 | 사람이 해야 했던 것 |
|---|---|
| 상황에 맞는 방법을 고름 | **무엇을 알고 싶은지** 말하기 |
| p값을 계산함 | **p값이 작다는 게 무슨 뜻인지** 판단 |
| 시킨 것을 함 | **데이터의 생김새를 알려 주기** (⑧) |

A 를 먼저 한 이유가 여기 있습니다.
**①~④ 를 손으로 해 봤기 때문에** ⑤~⑨ 의 답이 맞는지 알아볼 수 있는 것입니다.


---
# 직접 해 보기

이제 여러분 차례입니다. **아래 예시를 그대로 복사해 `ask("...")` 안에 넣고** 실행해 보세요.
그대로 써도 되고, 바꿔 써도 됩니다.

---

## 1. 더 캐물어 보기

앞에서 나온 답에 **한 번 더 파고드는** 질문입니다. 이게 가장 자주 쓰게 됩니다.

```
할인율을 구간별로 나눠서 (0%, 1~20%, 21~40%, 40% 초과)
구간마다 평균 이익과 주문 건수를 보여 줘.
어느 구간부터 이익이 마이너스로 돌아서는지도 말해 줘
```

```
이익이 가장 많이 나는 하위 카테고리 5개와
가장 많이 까먹는 하위 카테고리 5개를 뽑아 줘.
까먹는 쪽은 왜 그런지 짐작되는 이유도 데이터에서 찾아봐 줘
```

```
매출은 큰데 이익은 적은 고객이 있는지 찾아 줘.
그런 고객이 몇 명인지, 전체에서 어느 정도 비중인지도 알려 줘
```

---

## 2. 시간을 넣어 보기

**기간을 정하지 않으면 숫자가 거짓말을 합니다** &mdash; 재구매율 98.5% 가 그랬습니다.

```
연도별로 매출과 이익이 어떻게 변했는지 보여 주고,
좋아지고 있는지 나빠지고 있는지 한 문장으로 정리해 줘
```

```
2026년에 처음 산 고객이 그 해 안에 다시 산 비율을 구해 줘.
'재구매율'을 어떻게 정의했는지도 반드시 같이 말해 줘
```

```
월별 매출에 계절성이 있는지 확인해 줘.
있다면 어느 달이 높고 어느 달이 낮은지, 매년 반복되는 패턴인지도 봐 줘
```

---

## 3. 방법을 모를 때 &mdash; 그냥 물어보세요

**&ldquo;어떻게 하는지&rdquo; 대신 &ldquo;무엇을 알고 싶은지&rdquo;** 를 말하면 됩니다.

```
배송 방식(Ship Mode)에 따라 이익이 다른지 알고 싶어.
방식이 넷이라 두 개씩 비교하는 방법으로는 부족할 것 같은데,
적절한 방법을 골라서 확인하고 왜 그 방법인지도 말해 줘
```

```
고객 세그먼트(Segment)마다 주문 금액 분포가 다른지 알고 싶어.
평균만 비교하면 놓치는 게 있을 것 같으니 그것도 감안해서 봐 줘
```

```
할인을 몇 %까지 하면 이익이 남는지 알고 싶어.
이 데이터로 답할 수 있는 질문인지부터 판단하고,
답할 수 있다면 방법을 골라서 확인해 줘
```

---

## 4. 일부러 함정 던지기

**AI 가 어디서 넘어가는지** 보는 게 목적입니다. 오늘 배운 것을 확인하는 자리입니다.

```
이 데이터로 '할인을 줄이면 이익이 얼마나 늘어날지' 예측해 줘
```
&nbsp;&nbsp;&nbsp;&nbsp;→ 인과를 묻고 있습니다. **거절하거나 조건을 붙여야** 좋은 답입니다.

```
우리 회사에서 가장 중요한 고객이 누구야?
```
&nbsp;&nbsp;&nbsp;&nbsp;→ &lsquo;중요한&rsquo; 의 정의가 없습니다. **되물어야** 좋은 답입니다.

```
서부 지역은 이익률이 네 지역 중 가장 낮은데, 그 이유를 알려 줘
```
&nbsp;&nbsp;&nbsp;&nbsp;→ **전제가 거짓입니다.** ③에서 봤듯 서부는 이익률이 가장 **높습니다**(15.0%).
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;사실 확인부터 하고 **틀렸다고 말해 주는지** 보세요. 그냥 이유를 지어내면 나쁜 답입니다.

---

## 5. 보고할 수 있게 만들기

**숫자를 내는 것과 보고하는 것은 다릅니다.**

```
지금까지 확인한 것 중에서 영업기획 팀장에게 보고할 만한 것 세 가지를 골라,
각각 '무엇을 발견했나 / 그래서 무엇을 하자'로 정리해 줘
```

```
지역별 이익률을 막대그래프로 그려 줘.
제목에는 결론을 쓰고, 말하려는 지역만 다른 색으로 강조해 줘
```

```
방금 결론에서 '이 분석이 틀릴 수 있는 이유' 세 가지를 스스로 짚어 줘
```

---

## 여러분 회사 일로 바꿔 보기

이 표가 아니라 **여러분이 실제로 다루는 데이터**를 떠올리며 써 보세요.
2-9 절의 다섯 가지를 채우면 됩니다.

```
[상황]  저는 ___ 팀입니다. 지금 ___ 를 검토하고 있습니다.
[알고 싶은 것]  ___ 가 ___ 보다 ___ 한지 알고 싶습니다.
[데이터 형태]  한 행은 ___ 하나입니다. 기간은 ___ 입니다.
[원하는 형태]  표로 / 그래프로 / 세 문장 요약으로
[제약]  데이터는 올릴 수 없습니다. 파이썬 pandas 로 작업합니다.

그리고 이 분석이 맞지 않게 되는 경우도 함께 알려 주세요.
```

> 마지막 줄을 꼭 넣으세요.
> AI 는 대체로 자신 있게 답합니다. **이 질문을 해야 스스로 조건을 붙입니다.**

---

아래 셀에 넣고 실행하세요. **몇 번이든 바꿔 가며** 해 보시면 됩니다.


In [ ]:
ask("여기에 궁금한 것을 한국어로 적으세요")

---
# 정리 — 오늘 한 것

**Part 1 — 두 숫자의 관계**

1. **상관계수**는 -1 에서 +1 사이. 0에 가까우면 직선 관계가 약하다
2. 숫자만 보지 말고 **산점도를 함께** 본다
3. **상관은 인과가 아니다** — 순서가 반대일 수도, 숨은 원인이 있을 수도 있다

**Part 2 — 분석 에이전트**

4. **에이전트 = LLM + 하니스.** LangChain 의 `create_agent` 에 도구 하나와 맥락을 넣었다
5. **표를 통째로 보내지는 않는다.** 열 이름과 자료형만 보내도 코드가 나온다.
   다만 **계산 결과는 나간다** — AI 가 `print()` 한 것은 무엇이든 전송된다
6. AI 가 **거부하면** 질문이 잘못됐거나 데이터에 없다는 신호다
7. **코드가 맞아도 해석은 틀릴 수 있다** — 재구매율 98.5%
8. **p값이 작다 ≠ 차이가 크다.** 데이터가 많으면 작은 차이도 유의하게 나온다.
   검정을 시켰으면 **평균 차이가 몇 원인지**를 반드시 같이 본다

> 에이전트는 **코드를 대신 써 줍니다.**
> **무엇을 물을지, 나온 숫자가 말이 되는지**는 사람이 판단합니다.
